## Topic: Iterative Workflow in LangGraph

### Agenda:
- 1. Introduction of Iterative Workflow

- 2. Combining Iterative with Other Patterns

- 3. When to Use Iterative Workflows

- 4.  Key Takeaways

### 1. Introduction of Iterative Workflow

- Definition:
    - An Iterative Workflow is a graph pattern where one or more nodes repeat in a loop until a specific condition is met. Instead of moving forward to a new node, the graph loops back to an earlier node.
    
- The key idea is:
    - Do something → evaluate the result → if necessary, do it again → stop when the result is acceptable.

In [ ]:
""" 
Non-Iterative (Forward Only):
============================
  A → B → C → END
  Each node runs exactly once.

Iterative (With Loop):
======================
  A → B → C → (check) ──→ END  (if condition met)
                  ↑    │
                  │    └──→ B   (if condition NOT met, loop back!)
                  │         │
                  └─────────┘
  Node B and C may run 1, 2, 5, or 20 times.

"""


""" 
Real-World Analogy: Writing an Essay:
=====================================

  Non-Iterative Writer (One Draft):
  ┌───────┐    ┌───────┐    ┌───────┐
  │ Brain │───▶│ Write │───▶│ Submit│   ← Whatever you wrote, ship it!
  │ storm │    │ Draft │    │       │     (Probably full of errors)
  └───────┘    └───────┘    └───────┘

  Iterative Writer (Multiple Drafts):
  ┌───────┐    ┌───────┐    ┌───────┐    ┌───────┐
  │ Brain │───▶│ Write │───▶│Review │───▶│ Good? │──▶ Submit
  │ storm │    │ Draft │    │       │    │       │
  └───────┘    └───▲───┘    └───────┘    └───┬───┘
                   │                         │
                   │        No, fix it!      │
                   └─────────────────────────┘
                        (loop back!)

  The iterative writer produces BETTER work because
  they review and improve until the quality is right.

Main idea: 
    - Iterative workflows add repetition to our graph. The graph keeps looping through a set of nodes until a quality threshold, completion condition, or maximum retry count is reached.

    - In LangGraph, a loop is created by adding an edge that points backward to an earlier node__ Combined with a conditional edge, this creates a loop that repeats until a condition is met.

"""

In [ ]:
""" 
- Why Loops Matter in AI Agents:
=================================
┌─────────────────────────────────────────────────────────────┐
│           WHY LOOPS MATTER                                  │
│                                                             │
│  1. LLMS MAKE MISTAKES                                      │
│     The first answer is not always the best answer.         │
│     Loops let the agent try again with feedback.            │
│                                                             │
│  2. QUALITY REQUIRES ITERATION                              │
│     Great writing, code, and analysis come from             │
│     draft → review → revise → review → finalize.            │
│     One pass is rarely enough.                              │
│                                                             │
│  3. SELF-CORRECTION IS THE HALLMARK OF AGENTS               │
│     A chatbot gives one answer and stops.                   │
│     An agent checks its own work and fixes errors.          │
│     This REQUIRES a loop.                                   │
│                                                             │
│  4. REAL-WORLD TASKS ARE ITERATIVE                          │
│     • Software development: code → test → fix → test        │
│     • Research: search → read → refine query → search       │
│     • Writing: draft → edit → proofread → revise            │
│     • Design: prototype → feedback → iterate → prototype    │
│                                                             │
│  5. RETRIEVAL IS NOT ALWAYS PERFECT                         │
│     First retrieval may miss relevant documents.            │
│     Loop: retrieve → grade → refine query → retrieve again  │
└─────────────────────────────────────────────────────────────┘

"""

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           ANATOMY OF A LANGGRAPH LOOP                       │
│                                                             │
│  Every loop has 4 parts:                                    │
│                                                             │
│  1. ENTRY POINT                                             │
│     Where the loop begins. First iteration starts here.     │
│                                                             │
│  2. WORK NODE(S)                                            │
│     The node(s) that do the actual work each iteration.     │
│     Example: generate_answer, write_code, search_docs       │
│                                                             │
│  3. CHECK NODE / CONDITION                                  │
│     Evaluates whether the loop should continue or stop.     │
│     Example: grade_answer, run_tests, check_quality         │
│                                                             │
│  4. LOOP-BACK EDGE                                          │
│     The conditional edge that routes back to the work node  │
│     if the check fails, or forward to END if it passes.     │
│                                                             │
│  Template:                                                  │
│  ─────────                                                  │
│  START → [work] → [check] ──┬──→ END       (pass)           │
│                   ↑         └──→ [work]     (fail, retry)   │
│                   │                                         │
│                   └──────────────────────                   │
│                        (loop-back edge)                     │
└─────────────────────────────────────────────────────────────┘

"""

In [ ]:
""" 
-> The Most Important Safety Mechanism
    Every loop MUST have a maximum iteration limit. Without one, a failed condition could cause an infinite loop that runs forever and burns through your API credits.

WITHOUT GUARD (Dangerous!):
  generate → grade ──┬──→ END
        ↑            │
        └── (bad) ───┘
  If the LLM keeps producing bad answers, this loops FOREVER.
  - API costs: $∞

WITH GUARD (Safe!):
  generate → grade ──┬──→ END          (good)
        ↑            ├──→ END          (max retries reached!)
        └── (bad) ───┘
  After 3 retries, force exit even if quality is low.
  - API costs: bounded

"""


""" 
- Guard Rules:
==============
┌─────────────────────────────────────────────────────────────┐
│           ITERATION GUARD RULES                             │
│                                                             │
│  1. ALWAYS set a max_retries value                          │
│     Recommended: 3-5 for most use cases                     │
│                                                             │
│  2. Track the count in STATE                                │
│     Add retry_count: int to your TypedDict                  │
│                                                             │
│  3. Increment in the WORK NODE, not the check node          │
│     The work node runs each iteration, so it's the          │
│     right place to count.                                   │
│                                                             │
│  4. Check the count in the ROUTER FUNCTION                  │
│     if retry_count >= max_retries: return "done"            │
│                                                             │
│  5. Log when max retries is hit                             │
│     This helps with debugging and observability.            │
└─────────────────────────────────────────────────────────────┘

"""

### 2. Combining Iterative with Other Patterns

In [ ]:
""" 
- Real-World Graphs Use All Four Patterns Together:
====================================================

┌─────────────────────────────────────────────────────────────┐
│           COMBINING ALL FOUR PATTERNS                       │
│                                                             │
│  Example: Production CRAG Agent                             │
│                                                             │
│  START                                                      │
│    │                                                        │
│    ▼                                                        │
│  [Preprocess Query]          ← Sequential                   │
│    │                                                        │
│    ├──→ [Search Web]         ┐                              │
│    ├──→ [Search DB]          ├← Parallel                    │
│    └──→ [Search PDF]         ┘                              │
│         │                                                   │
│         ▼                                                   │
│  [Grade Documents]           ← Sequential                   │
│         │                                                   │
│    ┌────┴────┐                                              │
│    │         │               ← Conditional                  │
│  good      bad                                              │
│    │         │                                              │
│    ▼         ▼                                              │
│ [Generate] [Web Fallback]                                   │
│    │         │                                              │
│    └────┬────┘                                              │
│         ▼                                                   │
│  [Verify Answer]                                            │
│         │                                                   │
│    ┌────┴────┐                                              │
│    │         │               ← Conditional + Iterative      │
│  grounded  hallucination                                    │
│    │         │                                              │
│    ▼         └──→ [Regenerate] ──→ [Verify] (LOOP!)         │
│   END                                                       │
│                                                             │
│  This single graph uses:                                    │
│  -  Sequential (preprocess → grade)                         │
│  -  Parallel (search 3 sources)                             │
│  -  Conditional (good/bad routing)                          │
│  -  Iterative (regenerate until grounded)                   │
└─────────────────────────────────────────────────────────────┘


"""

### 3. When to Use Iterative Workflows

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           USE ITERATIVE WHEN...                             │
│                                                             │
│  - Quality matters more than speed                          │
│     The first attempt is rarely the best.                   │
│     Iterate until quality meets the threshold.              │
│                                                             │
│  - The task involves SELF-CORRECTION                        │
│     Generate → Check → Fix → Check → Done                   │
│     Examples: writing, coding, fact-checking                │
│                                                             │
│  - The task involves STEP-BY-STEP EXECUTION                 │
│     Plan → Execute step 1 → Execute step 2 → ... → Done     │
│     Examples: research, complex problem solving             │
│                                                             │
│  - The first retrieval may be INSUFFICIENT                  │
│     Retrieve → Grade → Refine query → Retrieve again        │
│     Examples: iterative RAG, adaptive search                │
│                                                             │
│  - The operation is UNRELIABLE (flaky APIs, networks)       │
│     Attempt → Fail → Retry → Succeed                        │
│     Examples: API calls, database connections               │
│                                                             │
│  REAL-WORLD EXAMPLES:                                       │
│  • Self-RAG (regenerate until factually grounded)           │
│  • Reflection agents (draft → critique → revise)            │
│  • Test-driven code generation (code → test → fix)          │
│  • Plan-and-execute agents (plan → step → step → done)      │
│  • Retry mechanisms (attempt → fail → retry → succeed)      │
│  • Iterative data cleaning (clean → validate → fix)         │
└─────────────────────────────────────────────────────────────┘


"""

### 4.  Key Takeaways

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│           ITERATIVE WORKFLOW IN LANGGRAPH                        │
│                                                                  │
│  WHAT:  A graph pattern where nodes repeat in a loop until       │
│         a condition is met (quality, completion, or max retries).│
│                                                                  │
│  PATTERN:                                                        │
│  ────────                                                        │
│  START → [work] → [check] ──┬──→ END       (condition met)       │
│                   ↑         └──→ [work]     (condition not met)  │
│                   │                                              │
│                   └────────────────────                          │
│                        (loop-back edge)                          │
│                                                                  │
│  HOW IT WORKS IN LANGGRAPH:                                      │
│  ──────────────────────────                                      │
│  1. Add a conditional edge after the check node                  │
│  2. One branch goes forward (to END or next step)                │
│  3. The other branch goes BACKWARD (to an earlier node)          │
│                                                                  │
│  graph.add_conditional_edges(                                    │
│      "check",                                                    │
│      should_continue,                                            │
│      {"done": END, "retry": "work_node"}  ← Backward edge!       │
│  )                                                               │
│                                                                  │
│  THE 4 PARTS OF EVERY LOOP:                                      │
│  ──────────────────────────                                      │
│  1. Entry Point    → Where the loop begins                       │
│  2. Work Node(s)   → Do the actual work each iteration           │
│  3. Check/Grade    → Evaluate if the loop should continue        │
│  4. Loop-Back Edge → Conditional edge pointing to earlier node   │
│                                                                  │
│  THE ITERATION GUARD (CRITICAL!):                                │
│  ────────────────────────────────                                │
│  • ALWAYS add retry_count: int to your state                     │
│  • ALWAYS increment it in the work node                          │
│  • ALWAYS check it in the router                                 │
│  • ALWAYS set a max (3-5 is typical)                             │
│                                                                  │
│  if state["retry_count"] >= state["max_retries"]:                │
│      return "done"  # Force exit!                                │
│                                                                  │
│  WITHOUT THIS GUARD, YOUR AGENT CAN LOOP FOREVER.                │
│                                                                  │
│  STATE ACROSS ITERATIONS:                                        │
│  ────────────────────────                                        │
│  • State persists across loop iterations                         │
│  • Each iteration reads previous iteration's output              │
│  • Use Annotated[list, operator.add] to keep history             │
│  • Or overwrite fields to keep only the latest version           │
│                                                                  │
│  COMMON ITERATIVE PATTERNS:                                      │
│  ──────────────────────────                                      │
│  Reflection:    generate → critique → revise → critique → done   │
│  Retry:         attempt → check → retry → check → done           │
│  Plan-Execute:  plan → step → step → step → synthesize → done    │
│  Iterative RAG: retrieve → generate → verify → retrieve → done   │
│  TDD Code:      write → test → fix → test → done                 │
│                                                                  │
│  WHEN TO USE:                                                    │
│  ─────────────                                                   │
│  • Quality > Speed                                               │
│  • Self-correction needed                                        │
│  • Step-by-step execution                                        │
│  • Unreliable operations (retry)                                 │
│  • Insufficient first retrieval                                  │
│                                                                  │
│  WHEN NOT TO USE:                                                │
│  ────────────────                                                │
│  • Simple one-shot tasks → SEQUENTIAL                            │
│  • Speed-critical → SEQUENTIAL or PARALLEL                       │
│  • Unreliable grader → Fix grader first                          │
│  • No clear "done" condition → Define metric first               │
│  • Cost-sensitive → Limit iterations strictly                    │
│                                                                  │
│  ALL FOUR PATTERNS SUMMARY:                                      │
│  ──────────────────────────                                      │
│  Sequential:  A → B → C → END          (fixed order)             │
│  Parallel:    A → [B,C,D] → E → END    (simultaneous)            │
│  Conditional: A → (if X) B / (if Y) C  (dynamic choice)          │
│  Iterative:   A → B → check → B → END  (loop until done)         │
│                                                                  │
│  Real graphs COMBINE all four patterns.                          │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Iteration is what separates good agents from great agents.     │
│   A chatbot gives one answer. An agent tries, checks,            │
│   and improves until the answer is RIGHT.                        │
│   But ALWAYS set a max iteration limit."                         │
└──────────────────────────────────────────────────────────────────┘

"""